In [3]:
#import libraries

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Introduction / Task Definition

This project will develop a machine learning system for classifying 27x27 RGB histopathology images of colon cells. The work has two prediction targets: a binary task that identifies whether a cell image is cancerous, and a multi-class task that predicts the cell type.

A key requirement is that the model must generalise to new patients, so the dataset will be split at the patient level rather than the image level. This prevents images from the same patient from appearing in both training and evaluation sets, which would otherwise create data leakage and inflate performance.

The overall workflow in this notebook follows a standard machine learning process:
define the task, 
inspect the dataset, 
perform exploratory analysis, 
preprocess the data, 
compare several models, 
tune hyperparameters, 
explore an advanced technique (CP), 
and finally evaluate the selected model on completely unseen patients.

## 2. Load Data / Inspection

In [7]:
DATA_ROOT = "dataset/data"

print("Current directory:", os.getcwd())
print("Dataset folder exists:", os.path.exists(DATA_ROOT))

print("\nFiles in dataset/data folder:")
for item in sorted(os.listdir(DATA_ROOT)):
    print(item)

main_path = os.path.join(DATA_ROOT, "data_labels_mainData.csv")
extra_path = os.path.join(DATA_ROOT, "data_labels_extraData.csv")

main_df = pd.read_csv(main_path)
extra_df = pd.read_csv(extra_path)

print("\nMain labels shape:", main_df.shape)
print("Extra labels shape:", extra_df.shape)

print("\nMain columns:", main_df.columns.tolist())
print("Extra columns:", extra_df.columns.tolist())

display(main_df.head())
display(extra_df.head())

Current directory: C:\Users\Owner\OneDrive - RMIT University\Documents\machinelearning2026\a3
Dataset folder exists: True

Files in dataset/data folder:
data_labels_extraData.csv
data_labels_mainData.csv
images

Main labels shape: (9896, 6)
Extra labels shape: (10384, 4)

Main columns: ['InstanceID', 'patientID', 'ImageName', 'cellTypeName', 'cellType', 'isCancerous']
Extra columns: ['InstanceID', 'patientID', 'ImageName', 'isCancerous']


,InstanceID,patientID,ImageName,cellTypeName,cellType,isCancerous
0,22405,1,22405.png,fibroblast,0,0
1,22406,1,22406.png,fibroblast,0,0
2,22407,1,22407.png,fibroblast,0,0
3,22408,1,22408.png,fibroblast,0,0
4,22409,1,22409.png,fibroblast,0,0


,InstanceID,patientID,ImageName,isCancerous
0,12681,61,12681.png,0
1,12682,61,12682.png,0
2,12683,61,12683.png,0
3,12684,61,12684.png,0
4,12685,61,12685.png,0


# 2. Notes
The dataset is loaded using relative paths from `dataset/data` so the notebook remains portable and reproducible. The two label files are read separately because the binary cancer task uses both files, while the cell-type task uses only the main label file.

The output shows that the dataset includes a `patientID` column for patient-level splitting, an `ImageName` column for linking labels to image files, and the target columns `isCancerous` and `cellTypeName`. I inspect the shapes and first rows of both label files to confirm the dataset structure before proceeding to splitting and EDA.